# Starter: A Simple Research Agent (free provider version)

The model decides what to search for, calls a web search tool (Tavily), reads the results, and may search again before writing a short sourced report.

This version uses the `openai` library, which works with several free providers. Pick one in Step 1.

**Before you run it**, create a `.env` file in this folder with the keys you need:

```
TAVILY_API_KEY=your_key_here
GROQ_API_KEY=your_key_here
GEMINI_API_KEY=your_key_here
```

You only need the key for the provider you choose, plus the Tavily key. Never share your keys or upload them anywhere public.

## Step 1: Choose a provider

Change `PROVIDER` to `'groq'` or `'gemini'`. Free tiers and model names change over time, so if you get a 'model not found' error, check the provider's dashboard for a current model name and update it here.

In [2]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from tavily import TavilyClient

load_dotenv()

PROVIDER = 'openrouter'   # options: 'groq', 'gemini', 'openrouter'

PROVIDERS = {
    'groq': {
        'base_url': 'https://api.groq.com/openai/v1',
        'key_name': 'GROQ_API_KEY',
        'model': 'llama-3.3-70b-versatile',
    },
    'gemini': {
        'base_url': 'https://generativelanguage.googleapis.com/v1beta/openai/',
        'key_name': 'GEMINI_API_KEY',
        'model': 'gemini-2.5-flash',
    },
    'openrouter': {
        'base_url': 'https://openrouter.ai/api/v1',
        'key_name': 'OPENROUTER_API_KEY',
        'model': 'openrouter/free',
    },
}

cfg = PROVIDERS[PROVIDER]
client = OpenAI(base_url=cfg['base_url'], api_key=os.environ[cfg['key_name']])
tavily = TavilyClient(api_key=os.environ['TAVILY_API_KEY'])
MODEL = cfg['model']
print('Using', PROVIDER, 'with model', MODEL)

Using openrouter with model openrouter/free


## Step 2: Define the tool

A tool is just a normal Python function. The agent can ask us to call it, and we send back whatever it returns.

In [3]:
def web_search(query, max_results=5):
    # Search the web and return a compact text summary of the results
    response = tavily.search(query=query, max_results=max_results)
    lines = []
    for r in response['results']:
        lines.append('- ' + r['title'] + ' (' + r['url'] + '): ' + r['content'][:400])
    return '\n'.join(lines)

# Quick test of the tool on its own (checks your Tavily key)
print(web_search('what is agentic AI'))

- What is Agentic AI? | IBM (https://www.ibm.com/think/topics/agentic-ai): Agentic AI is an artificial intelligence system that can accomplish a specific goal with limited supervision. It consists of AI agents—machine learning models that mimic human decision-making to solve problems in real time. In a multiagent system, each agent performs a specific subtask required to reach the goal and their efforts are coordinated through AI orchestration. [...] Unlike traditional A
- What is Agentic AI? | Salesforce (https://www.salesforce.com/agentforce/what-is-agentic-ai): Agentic AI is the technology that powers AI agents so they can operate autonomously without constant human oversight. It functions as a comprehensive platform that facilitates seamless interaction between AI agents and humans, fostering a collaborative environment. This platform includes a suite of tools and services to help AI agents learn, adapt, and collaborate to efficiently handle complex, dy
- What is Agentic AI? - Agen

## Step 3: Describe the tool to the model

The model can't see your Python code. It only sees this description, so a clear name and explanation help it decide when to use the tool.

In [4]:
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'web_search',
            'description': 'Search the web for current information on a topic. Returns titles, URLs and short snippets.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'query': {'type': 'string', 'description': 'The search query'}
                },
                'required': ['query']
            }
        }
    }
]

SYSTEM = (
    'You are a careful research assistant. Use the web_search tool as many times as you need. '
    'Then write a short report (under 300 words) and list your sources as URLs at the end.'
)

## Step 4: The agent loop

We keep calling the model. If it asks to use a tool, we run the tool, hand back the result, and call the model again. When it stops asking for tools, its answer is final. `max_steps` stops the agent from looping forever.

In [5]:
def run_research_agent(topic, max_steps=6):
    messages = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': 'Research this topic and write a short report with sources: ' + topic},
    ]

    for step in range(max_steps):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
        )
        msg = response.choices[0].message

        # No tool requested: the model has finished, so return its text
        if not msg.tool_calls:
            return msg.content

        # Keep the model's tool request in the conversation history
        messages.append({
            'role': 'assistant',
            'content': msg.content or '',
            'tool_calls': [
                {
                    'id': tc.id,
                    'type': 'function',
                    'function': {'name': tc.function.name, 'arguments': tc.function.arguments},
                }
                for tc in msg.tool_calls
            ],
        })

        # Run every requested tool and send the results back
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            print('Searching:', args['query'])
            result = web_search(args['query'])
            messages.append({
                'role': 'tool',
                'tool_call_id': tc.id,
                'content': result,
            })

    return 'Stopped: reached the maximum number of steps.'

## Step 5: Run it

Change the topic to anything you like. If you hit a rate limit error, wait a minute and run the cell again.

In [6]:
report = run_research_agent('How are companies using agentic AI workflows today?')
print()
print(report)

Searching: how companies are using agentic AI workflows today 2025

Companies are increasingly deploying agentic AI workflows to automate complex, multi-step processes with minimal human oversight. UIPath’s 2025 survey of IT leaders at enterprises exceeding $1 billion in revenue found 93% are highly interested in agentic AI for business workflows, highlighting strong adoption momentum. Boomi reports that 78% of organizations now use AI agents in some capacity, with 85% implementing them in at least one process—primarily for IT support, financial operations, and customer service. For instance, Legora’s Workflows enables legal teams to define objectives in natural language, triggering cascading AI tasks for document analysis and compliance workflows. Similarly, C3 AI’s agentic process automation replaces traditional RPA by allowing autonomous agents to manage end-to-end operational tasks. These systems enhance efficiency by dynamically chaining actions across applications, reducing manua

## Try next

1. **Reflection:** after the report is written, send it back to the model with the prompt 'Critique this report and improve it', then compare the two versions.
2. **A second tool:** add a tool that fetches a full web page, so the agent can read more than snippets.
3. **Evals:** write 3 test topics and check whether each report includes sources and stays under 300 words.
4. **Switch provider:** change `PROVIDER` in Step 1 and compare the reports.